# 03 load mysql

Carga dos dados no MySQL

In [17]:

%run 02_transform.ipynb

Conexão pronta para uso.
DIM_TEMPO: 1458 datas | 4 anos (2014-2017)
Produtos com categoria inconsistente: 0
DIM_PRODUTO: 1862 produtos | 3 categorias | 17 subcategorias
States únicos: 41
<StringArray>
[             'Alabama',              'Arizona',             'Arkansas',
           'California',             'Colorado',          'Connecticut',
             'Delaware', 'District of Columbia',              'Florida',
              'Georgia']
Length: 10, dtype: str
DIM_CLIENTE: 793 clientes |3 segmentos
 sk_envio modalidade_envio  prazo_medio_dias
        1     Second Class               3.2
        2   Standard Class               5.0
        3      First Class               2.2
        4         Same Day               0.0
FATO_VENDAS: 9,994 registros | Retornos: 800


In [18]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from config import engine
from sqlalchemy import text

print("Conexão pronta para uso.")

Conexão pronta para uso.


In [19]:
from sqlalchemy import create_engine, text
import sys, os

sys.path.insert(0, '..')
from config import DB_CONFIG

# Criar connection string
conn_str = (f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
            f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
            f"?charset=utf8mb4")

engine = create_engine(conn_str, echo=False)

# Testar conexão
with engine.connect() as conn:
    result = conn.execute(text('SELECT VERSION()')).fetchone()
    print(f'MySQL {result[0]} — conexão OK')

MySQL 8.0.46 — conexão OK


In [21]:
from sqlalchemy import text

# 1. Limpeza Total do Banco (Agora incluindo a DIM_GERENTE)
print('Limpando o banco de dados...')
with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE FATO_VENDAS;"))
    conn.execute(text("TRUNCATE TABLE DIM_TEMPO;"))
    conn.execute(text("TRUNCATE TABLE DIM_PRODUTO;"))
    conn.execute(text("TRUNCATE TABLE DIM_CLIENTE;"))
    conn.execute(text("TRUNCATE TABLE DIM_ENVIO;"))
    conn.execute(text("TRUNCATE TABLE DIM_GERENTE;")) # <-- Nova tabela adicionada na limpeza
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

# (Mantenha aqui os seus códigos de .str.slice() e cortes de texto, caso os use)
dim_produto['nome_produto'] = dim_produto['nome_produto'].astype(str).str.slice(0, 120)

# 2. Função de Carga
def load_table(df, table_name, engine, if_exists='append', index=False):
    df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)
    with engine.connect() as conn:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {table_name}')).scalar()
        print(f' {table_name:20}: {n:>7,} registros ca  rregados')

# 3. Execução da Carga (A ORDEM É FUNDAMENTAL)
print('\nCarregando dimensões...')
load_table(dim_tempo, 'DIM_TEMPO', engine)
load_table(dim_produto, 'DIM_PRODUTO', engine)
load_table(dim_cliente, 'DIM_CLIENTE', engine)
load_table(dim_envio, 'DIM_ENVIO', engine)
load_table(dim_gerente, 'DIM_GERENTE', engine) # <-- ESTA É A LINHA QUE FALTAVA!

print('\nCarregando tabela fato...')
load_table(fato_final, 'FATO_VENDAS', engine)
print('\nCarga concluída!')

Limpando o banco de dados...

Carregando dimensões...


C:\Users\GDK\AppData\Local\Temp\ipykernel_5296\4071318166.py:20: UserWarning: The provided table name 'DIM_TEMPO' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)
C:\Users\GDK\AppData\Local\Temp\ipykernel_5296\4071318166.py:20: UserWarning: The provided table name 'DIM_PRODUTO' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)


 DIM_TEMPO           :   1,458 registros ca  rregados
 DIM_PRODUTO         :   1,862 registros ca  rregados


C:\Users\GDK\AppData\Local\Temp\ipykernel_5296\4071318166.py:20: UserWarning: The provided table name 'DIM_CLIENTE' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)
C:\Users\GDK\AppData\Local\Temp\ipykernel_5296\4071318166.py:20: UserWarning: The provided table name 'DIM_ENVIO' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)
C:\Users\GDK\AppData\Local\Temp\ipykernel_5296\4071318166.py:20: UserWarning: The provided table name 'DIM_GERENTE' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_

 DIM_CLIENTE         :     793 registros ca  rregados
 DIM_ENVIO           :       4 registros ca  rregados
 DIM_GERENTE         :       4 registros ca  rregados

Carregando tabela fato...
 FATO_VENDAS         :   9,994 registros ca  rregados

Carga concluída!


C:\Users\GDK\AppData\Local\Temp\ipykernel_5296\4071318166.py:20: UserWarning: The provided table name 'FATO_VENDAS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)


In [22]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from config import engine
from sqlalchemy import text

print("Conexão pronta para uso.")

Conexão pronta para uso.


In [23]:
import pandas as pd
from sqlalchemy import create_engine, text
from config import engine
metas = pd.read_csv('../data/staging/metas_regionais.csv')
print(metas.to_string(index=False))

# Carregar no MySQL
metas.to_sql('DIM_META', engine, if_exists='append', index=False,method='multi', chunksize=100)
print('DIM_META carregada com sucesso!')

 regiao       categoria  meta_vendas  meta_lucro  ano_referencia
Central       Furniture        85000       12000            2014
   East       Furniture       120000       18000            2014
  South       Furniture        65000        9500            2014
   West       Furniture       140000       21000            2014
Central Office Supplies        95000       14000            2014
   East Office Supplies       135000       20000            2014
  South Office Supplies        72000       10800            2014
   West Office Supplies       160000       24000            2014
Central      Technology       110000       16500            2014
   East      Technology       145000       21750            2014
  South      Technology        88000       13200            2014
   West      Technology       175000       26250            2014


DatabaseError: Execution failed on sql 'INSERT INTO "DIM_META" (regiao, categoria, meta_vendas, meta_lucro, ano_referencia) VALUES (:regiao_m0, :categoria_m0, :meta_vendas_m0, :meta_lucro_m0, :ano_referencia_m0), (:regiao_m1, :categoria_m1, :meta_vendas_m1, :meta_lucro_m1, :ano_referencia_m1), (:regiao_m2, :categoria_m2, :meta_vendas_m2, :meta_lucro_m2, :ano_referencia_m2), (:regiao_m3, :categoria_m3, :meta_vendas_m3, :meta_lucro_m3, :ano_referencia_m3), (:regiao_m4, :categoria_m4, :meta_vendas_m4, :meta_lucro_m4, :ano_referencia_m4), (:regiao_m5, :categoria_m5, :meta_vendas_m5, :meta_lucro_m5, :ano_referencia_m5), (:regiao_m6, :categoria_m6, :meta_vendas_m6, :meta_lucro_m6, :ano_referencia_m6), (:regiao_m7, :categoria_m7, :meta_vendas_m7, :meta_lucro_m7, :ano_referencia_m7), (:regiao_m8, :categoria_m8, :meta_vendas_m8, :meta_lucro_m8, :ano_referencia_m8), (:regiao_m9, :categoria_m9, :meta_vendas_m9, :meta_lucro_m9, :ano_referencia_m9), (:regiao_m10, :categoria_m10, :meta_vendas_m10, :